In [0]:
from pyspark.sql.functions import col, rank
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("table_name", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
table_name = dbutils.widgets.get("table_name")

In [0]:
df_silver = spark.read.table(f"{catalog}.{schema}.{table_name}")

In [0]:
df_silver_agg = (df_silver
 .withColumn("dt_trip", col("ts_trip_pickup").cast("date"))
 .groupBy("dt_trip", "ct_vendor_name")
 .agg({"trip_duration_in_min": "sum",
       "total_amount": "sum",
       "passager_count": "sum",
       "trip_distance": "sum",
       "pk_trip": "count"
    })
 .orderBy("dt_trip")
)

In [0]:
window_dt_trip_vendor = Window.partitionBy("dt_trip", "ct_vendor_name").orderBy(col("count").desc())

df_most_payment_usage = (df_silver
    .withColumn("dt_trip", col("ts_trip_pickup").cast("date"))
    .groupBy("dt_trip", "ct_vendor_name", "ct_payment_type")
    .count()
    .withColumn("rank", rank().over(window_dt_trip_vendor))
    .filter(col("rank") == 1)
    .drop("rank", "count")
)

In [0]:
df_gold = (df_silver_agg
    .join(df_most_payment_usage, on=["dt_trip", "ct_vendor_name"], how="inner")
    .select(
        col("dt_trip"),
        col("ct_vendor_name"),
        col("ct_payment_type").alias("most_payment_usage"),
        col("sum(trip_duration_in_min)").alias("total_trip_duration"),
        col("sum(total_amount)").alias("total_amount"),
        col("sum(passager_count)").alias("total_passenger"),
        col("sum(trip_distance)").alias("total_distance"),
        col("count(pk_trip)").alias("total_trips")    
    )
)

In [0]:
df_gold.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.gold_trip_data")